In [ ]:
import omero_analysis_notebook as oan
ctx = oan.configure(r'''{"schema": "nl.bioimaging.omero-analysis-notebook.v1", "inputs": [{"id": "measurements", "kind": "query", "path": "input/measurements.duckdb", "formats": ["duckdb"], "required": true}], "results": {"path": "results"}, "parameters": [], "requirements": ["pandas"]}''')
ctx.display_parameters()


In [ ]:
import pandas as pd
schema = await ctx.query("measurements", "SELECT table_name FROM information_schema.tables WHERE table_schema = 'main' AND table_type = 'BASE TABLE' ORDER BY table_name")
rows = []
for table in schema["table_name"].tolist():
    quoted = '"' + table.replace('"', '""') + '"'
    result = await ctx.query("measurements", "SELECT COUNT(*) AS row_count FROM " + quoted)
    rows.append({"table_name": table, "row_count": int(result.iloc[0]["row_count"])})
summary = pd.DataFrame(rows)
summary.to_csv(ctx.results / "cross-user-table-counts.csv", index=False)
print(summary.to_string(index=False))
